# DEH 30-Day PySpark Challenge
## Day 4 — DataFrame Writer & Write Modes

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## Step 4 — Read orders with explicit schema

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Rows loaded: {orders_df.count()}")
orders_df.show(3)

## Step 5 — Write modes

In [ ]:
# overwrite — deletes existing data and writes fresh
orders_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"s3a://{BUCKET}/output/orders_csv/")

print("Written with overwrite mode")

# Run again — overwrite means no error
orders_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"s3a://{BUCKET}/output/orders_csv/")

print("Written again — overwrite replaced the previous output")

In [ ]:
# append — adds to existing data
orders_df.write \
    .mode("append") \
    .option("header", "true") \
    .csv(f"s3a://{BUCKET}/output/orders_append/")

# Run again — appends more data
orders_df.write \
    .mode("append") \
    .option("header", "true") \
    .csv(f"s3a://{BUCKET}/output/orders_append/")

# Read back — will have 200 rows (100 written twice)
appended = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/output/orders_append/")

print(f"Rows after two appends: {appended.count()}")

## Step 6 — Writing Parquet

In [ ]:
# Write as Parquet — no options needed
orders_df.write \
    .mode("overwrite") \
    .parquet(f"s3a://{BUCKET}/output/orders_parquet/")

print("Parquet written")

# Read back — no schema or header needed, schema is embedded
df_parquet = spark.read.parquet(f"s3a://{BUCKET}/output/orders_parquet/")
df_parquet.printSchema()
df_parquet.show(3)

## Step 7 — Partitioning on write

In [ ]:
# Write partitioned by region
orders_df.write \
    .mode("overwrite") \
    .partitionBy("region") \
    .parquet(f"s3a://{BUCKET}/output/orders_partitioned/")

print("Partitioned Parquet written")
print("S3 structure:")
print("  output/orders_partitioned/region=East/")
print("  output/orders_partitioned/region=West/")
print("  output/orders_partitioned/region=South/")
print("  output/orders_partitioned/region=Midwest/")

In [ ]:
# Read back — filter by partition (Spark reads ONLY that folder)
east_df = spark.read \
    .parquet(f"s3a://{BUCKET}/output/orders_partitioned/") \
    .filter("region = 'East'")

print(f"East partition rows: {east_df.count()}")
east_df.show(5)

## Step 8 — Verify write output

In [ ]:
# Always read back to verify
verify_df = spark.read.parquet(f"s3a://{BUCKET}/output/orders_parquet/")

print(f"Rows written    : {orders_df.count()}")
print(f"Rows read back  : {verify_df.count()}")
print(f"Match           : {orders_df.count() == verify_df.count()}")

---
## Assignment — Day 4

---

### Task 1
Read `customers.csv` from S3 with an explicit schema.  
Write it to S3 as CSV using `overwrite` mode with a header.  
Read it back and verify the row count matches.

In [ ]:
# Task 1 — Write your code here


### Task 2
Read `orders.csv` from S3 with an explicit schema.  
Write it as Parquet to `output/orders_parquet/`.  
Read the Parquet back — notice no schema or header option needed.  
Print the schema of the Parquet file.

In [ ]:
# Task 2 — Write your code here


### Task 3
Write `orders.csv` to S3 as Parquet partitioned by `region`.  
Read back only the `East` region partition.  
How many rows does the East partition have?

In [ ]:
# Task 3 — Write your code here


### Task 4
Try writing to the same Parquet path twice without specifying a mode.  
What error do you get?  
Then fix it by adding the correct write mode.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*